# Anchor-field tilt ("Anchor Strength") validation sweep

The completer had a diagnosed **"sore thumb" failure**: pin a niche Pokémon
(the canonical case is Camerupt) and the PT completer sometimes returns a
coherent 5-core with the pin bolted on, contributing little or negative
coupling. The pairwise energy is a **sum over pairs**, so a strong core can
carry a poorly-integrated sixth and still post a high total; the conditional
`P(team | pin)` is multimodal, and the pin-centric mode is often not the
top one — which also makes top-1-by-frequency **seed-unstable**.

The fix is **exponential tilting on the pin-integration statistic**
`T(s) = Σ_{p∈pins} Σ_{j free} J[p,j]·s_j`:

$$H_\alpha = H - (\alpha - 1)\,T(s)$$

Because the pins are fixed, this is just a shift of the effective field
(`h_eff[j] += (α−1)·Σ_p J[p,j]` for free candidates), i.e. a genuine
Boltzmann tilt, not a rerank. `P_α` is a one-parameter exponential family in
α with sufficient statistic `T`, so

$$\frac{d}{d\alpha}\,\mathbb{E}_\alpha[T] = \mathrm{Var}_\alpha(T) \ge 0,$$

and the sweep's monotonicity check below is a **theorem**, not a hope — a
dip beyond Monte-Carlo noise is a wiring bug.

Correctness (PT-vs-exact-enumeration of the tilted conditional, α=1 no-op,
greedy/MF ≡ boosted-field, TS/Python parity) is gated in
`tests/test_sampling.py::TestAnchorFieldTilt` and `tests/test_parity.py`.
This notebook does the **product validation** on the real fitted model:

1. mean pin-integration is monotone in α for every pin;
2. the top-1 completion becomes pin-centric and **seed-stable**;
3. what α costs — raw (untilted) score and variety vs α;
4. a popular-pin control (the tilt should be harmless when the pin needs
   no help);
5. the team gallery behind the shipped defaults (`DEFAULT_ANCHOR = 1` on
   /completer, `ANCHOR_ARTICLE_DEFAULT = 2.0` in the article MiniCompleter).

In [1]:
from __future__ import annotations

from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from k2dex.potts import load_fitted_model
from k2dex.sampling import parallel_tempered_mcmc

# Resolve the committed artifact whether the kernel cwd is the repo root or
# notebooks/. reg-m-a is the regulation the sore thumb was diagnosed in.
_candidates = [Path("web/public/models/reg-m-a"), Path("../web/public/models/reg-m-a")]
MODEL_DIR = next(p for p in _candidates if p.exists())

model = load_fitted_model(MODEL_DIR)
V = len(model.h)
print(f"model: {MODEL_DIR}  V={V}  corpus teams={model.n_corpus_teams}")

model: web/public/models/reg-m-a  V=843  corpus teams=22037


## Pins

Four niche pins (the diagnosed failure cases) and one popular control.
Each pin is the species' **highest-marginal feature** — exactly what the
webapp's `resolveSitePins` seeds when you pin a species without choosing an
item. The Python sampler has feature pins only, so this mirrors the webapp's
site-pin behavior up to the item reroll.

In [2]:
NICHE = ["Camerupt", "Vivillon", "Torkoal", "Crabominable"]
POPULAR = ["Incineroar"]

def pin_feature(species: str) -> int:
    feats = [i for i, s in enumerate(model.species_of) if s == species]
    if not feats:
        raise ValueError(f"species not in vocab: {species}")
    return max(feats, key=lambda i: model.m[i])

PINS = {sp: pin_feature(sp) for sp in NICHE + POPULAR}
print(f"{'species':16s} {'pin feature':32s} {'m':>7} {'h':>7}")
for sp, i in PINS.items():
    print(f"{sp:16s} {model.vocab[i]:32s} {model.m[i]:7.4f} {model.h[i]:7.3f}")

species          pin feature                            m       h
Camerupt         Camerupt @ Cameruptite            0.0150  -0.626
Vivillon         Vivillon @ Focus Sash             0.0166  -0.781
Torkoal          Torkoal @ Charcoal                0.0468  -0.496
Crabominable     Crabominable @ Crabominite        0.0131  -0.676
Incineroar       Incineroar @ Sitrus Berry         0.1737   1.210


## Sweep machinery

One PT completion per `(pin, α, seed)` at the webapp operating point
(`field_weight = 1`, cold `T = 1`). `QUICK = True` uses a trimmed budget
(~2 s/run, plenty for the sweep's statistics); flip it off for the full
production ladder/sweeps if you want the exact /completer budget.

Per run we record:

- **meanT** — mean pin-integration `Σ_{j on} J[pin, j]` over the cold-chain
  samples (the tilted statistic; must rise with α);
- **top-1 team** by frequency, its **share** of samples, its **raw**
  (untilted) score `h·s + ½ s'Js` and its own pin-integration;
- **variety** — mean pairwise differing members across the top-10 teams
  (the diversity α spends).

In [3]:
QUICK = True
if QUICK:
    T_LADDER = np.geomspace(1.0, 3.0, 5)
    N_STEPS, BURN_IN = 12_000, 3_000
else:  # the /completer production budget (constants.ts PT_* per run)
    T_LADDER = np.geomspace(1.0, 3.0, 7)
    N_STEPS, BURN_IN = 25_000, 5_000
SWAP_INTERVAL = 10

ALPHAS = [1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
SEEDS = [11, 12, 13]


def run_one(pin: int, alpha: float, seed: int) -> dict:
    res = parallel_tempered_mcmc(
        model.J, model.h, model.team_size, [pin], [], 1.0,
        T_LADDER, N_STEPS, BURN_IN, SWAP_INTERVAL, seed,
        species_of=model.species_of, item_of=model.item_of,
        anchor_strength=alpha,
    )
    assert res is not None
    samples, _, _ = res

    Jrow = model.J[pin]
    meanT = float((samples @ Jrow).mean())  # J[pin,pin]=0, so the pin's own bit is free

    counts: Counter[tuple[int, ...]] = Counter(
        tuple(np.nonzero(row)[0]) for row in samples
    )
    top = counts.most_common(10)
    top1, top1_n = top[0]
    variety = (
        float(np.mean([len(a - b) for a, b in combinations([set(t) for t, _ in top], 2)]))
        if len(top) > 1 else 0.0
    )
    s = np.zeros(V)
    s[list(top1)] = 1.0
    return {
        "meanT": meanT,
        "top1": top1,
        "top1_share": top1_n / len(samples),
        "top1_T": float(sum(Jrow[j] for j in top1 if j != pin)),
        "top1_raw": float(model.h @ s + 0.5 * s @ model.J @ s),
        "variety": variety,
    }

In [4]:
results: dict[tuple[str, float, int], dict] = {}
grid = [(sp, a, sd) for sp in PINS for a in ALPHAS for sd in SEEDS]
for sp, a, sd in tqdm(grid, desc="anchor sweep"):
    results[(sp, a, sd)] = run_one(PINS[sp], a, sd)
print(f"{len(results)} runs")

anchor sweep:   0%|          | 0/105 [00:00<?, ?it/s]

105 runs


## 1. Monotonicity: mean pin-integration vs α

Seed-averaged `E[T]` per pin. The exponential-family identity guarantees the
true curves are non-decreasing; the sampled ones should be too, up to MC
noise (the shaded band is the seed min–max). The curves also flatten as α
grows — `dE[T]/dα = Var(T)`, and the tilt itself shrinks that variance, so
the knob self-limits.

**The noise allowance must be estimated, not fixed.** The slope and the
sampling noise are the *same quantity*: `dE[T]/dα = Var(T)`, and the MC
error of each point also scales with `Var(T)` (divided by the effective
sample size, which shrinks in the crossover region where the chain hops
between modes). So the steepest part of the curve is also the noisiest —
a fixed threshold is tightest exactly where scatter is worst. (Empirically:
Camerupt's 1.5→1.75 step showed a −0.17 "dip" on one 3-seed run; a 12-seed
recheck put the true step at **+1.0** with per-seed sd ≈ 0.5–0.75, i.e. the
dip was well within 1σ of an adjacent-α difference.) The check below
tolerates `3·SE(diff)` from the seed-level SEMs, floored at 0.3 because a
3-seed sd estimate is itself noisy. If a dip exceeds that, rerun with more
`SEEDS` before suspecting the sampler — a real wiring bug shows up as a
*persistent* violation that grows with sample size, not a one-off.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for sp in PINS:
    m = np.array([[results[(sp, a, sd)]["meanT"] for sd in SEEDS] for a in ALPHAS])
    style = dict(ls="--", alpha=0.9) if sp in POPULAR else {}
    (line,) = ax.plot(ALPHAS, m.mean(axis=1), marker="o", label=sp, **style)
    ax.fill_between(ALPHAS, m.min(axis=1), m.max(axis=1), color=line.get_color(), alpha=0.15)
ax.set_xlabel("Anchor Strength α")
ax.set_ylabel("mean pin-integration  E[T]")
ax.set_title("E[T] is monotone in α (theorem; a dip beyond noise = wiring bug)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

# Noise-aware monotonicity check: each adjacent-α difference gets a tolerance
# of 3·SE(diff) from the seed-level SEMs (floored — a len(SEEDS)-sample sd is
# itself a noisy estimate). See the markdown above for why a fixed threshold
# is wrong here.
SE_FLOOR = 0.3
for sp in PINS:
    m = np.array([[results[(sp, a, sd)]["meanT"] for sd in SEEDS] for a in ALPHAS])
    curve = m.mean(axis=1)
    sem = m.std(axis=1, ddof=1) / np.sqrt(len(SEEDS))
    diffs = np.diff(curve)
    tol = np.maximum(SE_FLOOR, 3.0 * np.sqrt(sem[:-1] ** 2 + sem[1:] ** 2))
    bad = diffs < -tol
    assert not bad.any(), (
        f"non-monotone beyond 3·SE for {sp}: diffs={np.round(diffs, 3)}, "
        f"tol={np.round(tol, 3)} — rerun with more SEEDS before suspecting the sampler"
    )
print("monotonicity: OK for all pins (within 3·SE of the seed scatter)")

## 2. Seed stability of the top-1 completion

The user-facing symptom of multimodality: at α = 1 different seeds return
different top-1 teams for a niche pin. Count distinct top-1 teams across the
seeds (1 = stable). Note the article MiniCompleter aggregates 10 independent
runs, so single-run stability here is a *conservative* proxy — and the top-1
**share** is the graded version of the same signal (a decisive mode
dominates the samples).

In [ ]:
print(f"{'species':16s}" + "".join(f"  α={a:<5}" for a in ALPHAS))
print("distinct top-1 teams across seeds (1 = stable)")
for sp in PINS:
    ks = [len({results[(sp, a, sd)]["top1"] for sd in SEEDS}) for a in ALPHAS]
    print(f"{sp:16s}" + "".join(f"  {k:<7}" for k in ks))
print("\nmean top-1 share of samples")
for sp in PINS:
    sh = [np.mean([results[(sp, a, sd)]["top1_share"] for sd in SEEDS]) for a in ALPHAS]
    print(f"{sp:16s}" + "".join(f"  {s:<7.1%}" for s in sh))

## 3. What α costs: raw score and variety

The tilt changes *which* teams are found, and we report their **untilted**
score (the webapp never displays tilted energies). For niche pins the raw
score of the top-1 should *rise* through moderate α — the tilt digs the
genuinely-good pin-centric mode out from under the generic cores — then
fall once α starts strong-arming fringe partners onto the team. Variety
declines monotonically; we want a value that stabilizes the pin without
flattening the distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for sp in PINS:
    raw = [np.mean([results[(sp, a, sd)]["top1_raw"] for sd in SEEDS]) for a in ALPHAS]
    var = [np.mean([results[(sp, a, sd)]["variety"] for sd in SEEDS]) for a in ALPHAS]
    style = dict(ls="--", alpha=0.9) if sp in POPULAR else {}
    axes[0].plot(ALPHAS, raw, marker="o", label=sp, **style)
    axes[1].plot(ALPHAS, var, marker="o", label=sp, **style)
axes[0].set_xlabel("α"); axes[0].set_ylabel("top-1 raw score (untilted)")
axes[0].set_title("Raw score of the returned team")
axes[1].set_xlabel("α"); axes[1].set_ylabel("top-10 variety (mean pairwise diff)")
axes[1].set_title("Diversity cost")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()

## 4. The team gallery: Camerupt, α = 1 vs α = 2

The motivating case, member by member. Each row shows the member's coupling
to the pin — at α = 1 look for the sore thumb (the pin's own row is the
only weak link on an otherwise-coherent team, or the pin's couplings are
simply small); at α = 2 the team should be visibly built *around* the pin.

In [ ]:
def show_team(sp: str, alpha: float, seed: int) -> None:
    pin = PINS[sp]
    r = results[(sp, alpha, seed)]
    print(f"{sp} @ α={alpha} (seed {seed})  "
          f"raw={r['top1_raw']:+.2f}  T={r['top1_T']:+.2f}  share={r['top1_share']:.1%}")
    for j in sorted(r["top1"], key=lambda j: -model.J[pin][j]):
        tag = " <- pin" if j == pin else f"  J[pin,·]={model.J[pin][j]:+.3f}"
        print(f"    {model.vocab[j]:36s}{tag}")
    print()

for sd in SEEDS:
    show_team("Camerupt", 1.0, sd)
for sd in SEEDS[:1]:
    show_team("Camerupt", 2.0, sd)

## 5. Popular-pin control

Incineroar needs no help — the check is that α = 2 doesn't *hurt*: the
returned team's raw score should be at least in the same range as α = 1
(in the original sweep it actually rose, because even a popular pin's
conditional carries some passenger modes).

In [ ]:
for a in (1.0, 2.0):
    raws = [results[("Incineroar", a, sd)]["top1_raw"] for sd in SEEDS]
    print(f"α={a}: top-1 raw scores across seeds = "
          + ", ".join(f"{r:+.2f}" for r in raws))
show_team("Incineroar", 2.0, SEEDS[0])

## Verdict

From the original tuning sweep (and reproducible above):

- **Monotonicity holds for every pin** — the implementation samples the
  tilted family it claims to.
- **α = 1** reproduces the sore thumb: niche pins get low integration and
  seed-unstable top-1 teams (Camerupt: T ≈ +2.8, different team per seed).
- **α = 2** is the sweet spot: decisively pin-centric top-1 (shares jump
  from ~2% to 10–40%), *higher* raw scores for every niche pin, variety
  intact, and the popular control undamaged — shipped as
  `ANCHOR_ARTICLE_DEFAULT = 2.0` (`web/src/constants.ts`).
- **α = 3** overshoots: Torkoal's raw score drops below its α = 1 baseline
  (fringe Trick-Room partners get strong-armed in) and Vivillon's variety
  collapses.
- The main completer ships `DEFAULT_ANCHOR = 1` (neutral) with the
  "Anchor Strength" slider exposed: the tilt answers a different question
  ("build around my picks") than the model's conditional ("what do teams
  with these picks look like"), so it's opt-in there and default-on only in
  the article's build-around demo.